# 09 — Official-Style MS-TCN Text Teacher + KD on Breakfast

Next stage after the official MS-TCN visual-only baseline.

Goal:

```text
Train a text-aware teacher that sees video + text during training,
then distill it into a video-only student for inference.
```

Important wording:

- This is **official-style / official-compatible MS-TCN**.
- It uses the same Breakfast split, features, mapping, and TAS metrics as the official baseline.
- It is self-contained because the original official MS-TCN `Trainer` is visual-only; adding text/KD cleanly is easier here.

Default full run:

```text
CE video-only student: 10 epochs
concat text teacher:   20 epochs
KD video-only student: 15 epochs
```

Final inference model:

```text
student_kd_from_text_teacher
```

It uses **video only** at inference.

## 1. Mount Drive, imports, and device check

In [14]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, json, time, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: CPU detected. Do not run full training on CPU.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device: cuda
GPU: Tesla T4


## 2. Paths

In [15]:
DRIVE_ROOT = Path('/content/drive/MyDrive/mmf_tas_lab_data')

DRIVE_BREAKFAST_ROOT = DRIVE_ROOT / 'zenodo_ms_tcn_data' / 'breakfast'
DRIVE_FEATURE_DIR = DRIVE_BREAKFAST_ROOT / 'features'
DRIVE_GT_DIR = DRIVE_BREAKFAST_ROOT / 'groundTruth'
DRIVE_SPLIT_DIR = DRIVE_BREAKFAST_ROOT / 'splits'
DRIVE_MAPPING_PATH = DRIVE_BREAKFAST_ROOT / 'mapping.txt'

TEXT_ROOT = DRIVE_ROOT / 'text_assisted_tas' / 'breakfast' / 'text_embeddings'
TEXT_EMB_PATH = TEXT_ROOT / 'breakfast_clip_vitb16_text_embeddings.npy'
TEXT_META_PATH = TEXT_ROOT / 'breakfast_clip_vitb16_text_embedding_metadata.csv'

LOCAL_BREAKFAST_ROOT = Path('/content/breakfast_local')
LOCAL_FEATURE_DIR = LOCAL_BREAKFAST_ROOT / 'features'
LOCAL_GT_DIR = LOCAL_BREAKFAST_ROOT / 'groundTruth'
LOCAL_SPLIT_DIR = LOCAL_BREAKFAST_ROOT / 'splits'
LOCAL_MAPPING_PATH = LOCAL_BREAKFAST_ROOT / 'mapping.txt'

RUN_BASE = DRIVE_ROOT / 'text_assisted_tas' / 'breakfast' / 'runs'

for p in [DRIVE_FEATURE_DIR, DRIVE_GT_DIR, DRIVE_SPLIT_DIR, DRIVE_MAPPING_PATH, TEXT_EMB_PATH]:
    assert Path(p).exists(), p

print('Drive Breakfast:', DRIVE_BREAKFAST_ROOT)
print('Text embeddings:', TEXT_EMB_PATH)
print('Run base:', RUN_BASE)

Drive Breakfast: /content/drive/MyDrive/mmf_tas_lab_data/zenodo_ms_tcn_data/breakfast
Text embeddings: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/text_embeddings/breakfast_clip_vitb16_text_embeddings.npy
Run base: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs


## 3. Copy Breakfast dataset to local Colab disk

In [16]:
EXPECTED_FEATURES = 1712
EXPECTED_GT = 1712

def local_copy_is_complete():
    if not LOCAL_FEATURE_DIR.exists() or not LOCAL_GT_DIR.exists() or not LOCAL_SPLIT_DIR.exists():
        return False
    n_feat = len(list(LOCAL_FEATURE_DIR.glob('*.npy')))
    n_gt = len(list(LOCAL_GT_DIR.glob('*.txt')))
    return n_feat == EXPECTED_FEATURES and n_gt == EXPECTED_GT and LOCAL_MAPPING_PATH.exists()

if local_copy_is_complete():
    print('Local dataset already complete. Skipping copy.')
else:
    if LOCAL_BREAKFAST_ROOT.exists():
        print('Removing incomplete local copy:', LOCAL_BREAKFAST_ROOT)
        shutil.rmtree(LOCAL_BREAKFAST_ROOT)
    LOCAL_BREAKFAST_ROOT.mkdir(parents=True, exist_ok=True)

    print('Copying features. This can take several minutes...')
    shutil.copytree(DRIVE_FEATURE_DIR, LOCAL_FEATURE_DIR)
    print('Copying groundTruth...')
    shutil.copytree(DRIVE_GT_DIR, LOCAL_GT_DIR)
    print('Copying splits...')
    shutil.copytree(DRIVE_SPLIT_DIR, LOCAL_SPLIT_DIR)
    print('Copying mapping.txt...')
    shutil.copy2(DRIVE_MAPPING_PATH, LOCAL_MAPPING_PATH)

print('local features:', len(list(LOCAL_FEATURE_DIR.glob('*.npy'))))
print('local groundTruth:', len(list(LOCAL_GT_DIR.glob('*.txt'))))
print('local splits:', len(list(LOCAL_SPLIT_DIR.glob('*'))))
print('local mapping:', LOCAL_MAPPING_PATH.exists())

assert len(list(LOCAL_FEATURE_DIR.glob('*.npy'))) == EXPECTED_FEATURES
assert len(list(LOCAL_GT_DIR.glob('*.txt'))) == EXPECTED_GT
assert LOCAL_MAPPING_PATH.exists()

Local dataset already complete. Skipping copy.
local features: 1712
local groundTruth: 1712
local splits: 9
local mapping: True


## 4. Experiment configuration

In [17]:
#RUN_MODE = 'smoke_debug'
RUN_MODE = 'full_split1_text_kd'

SPLIT_ID = 1

CONFIGS = {
    'smoke_debug': {
        'max_train_videos': 20,
        'max_test_videos': 20,
        'epochs_ce': 2,
        'epochs_teacher': 2,
        'epochs_kd': 2,
        'num_stages': 4,
        'num_layers': 10,
        'num_f_maps': 64,
    },
    'full_split1_text_kd': {
        'max_train_videos': None,
        'max_test_videos': None,
        'epochs_ce': 10,
        'epochs_teacher': 20,
        'epochs_kd': 15,
        'num_stages': 4,
        'num_layers': 10,
        'num_f_maps': 64,
    },
}

cfg = CONFIGS[RUN_MODE]
MAX_TRAIN_VIDEOS = cfg['max_train_videos']
MAX_TEST_VIDEOS = cfg['max_test_videos']
EPOCHS_CE = cfg['epochs_ce']
EPOCHS_TEACHER = cfg['epochs_teacher']
EPOCHS_KD = cfg['epochs_kd']
NUM_STAGES = cfg['num_stages']
NUM_LAYERS = cfg['num_layers']
NUM_F_MAPS = cfg['num_f_maps']

BATCH_SIZE = 1
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5
KD_TEMPERATURE = 4.0
LAMBDA_CE = 1.0
LAMBDA_KD = 0.05

RUN_NAME = f'official_style_mstcn_text_kd_{RUN_MODE}_split{SPLIT_ID}'
RUN_ROOT = RUN_BASE / RUN_NAME
CHECKPOINT_DIR = RUN_ROOT / 'checkpoints'
PRED_DIR = RUN_ROOT / 'predictions'
EVAL_DIR = RUN_ROOT / 'evaluation_metrics'
for p in [RUN_ROOT, CHECKPOINT_DIR, PRED_DIR, EVAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# if RUN_MODE.startswith('full') and device.type != 'cuda':
#     raise RuntimeError('Full training requested, but CUDA is not available. Switch Colab runtime to GPU.')

config_to_save = {
    'run_mode': RUN_MODE,
    'run_name': RUN_NAME,
    'split_id': SPLIT_ID,
    'max_train_videos': MAX_TRAIN_VIDEOS,
    'max_test_videos': MAX_TEST_VIDEOS,
    'epochs_ce': EPOCHS_CE,
    'epochs_teacher': EPOCHS_TEACHER,
    'epochs_kd': EPOCHS_KD,
    'num_stages': NUM_STAGES,
    'num_layers': NUM_LAYERS,
    'num_f_maps': NUM_F_MAPS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'lambda_ce': LAMBDA_CE,
    'lambda_kd': LAMBDA_KD,
    'kd_temperature': KD_TEMPERATURE,
    'device': str(device),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'run_root': str(RUN_ROOT),
}
(RUN_ROOT / 'config.json').write_text(json.dumps(config_to_save, indent=2))
print(json.dumps(config_to_save, indent=2))

{
  "run_mode": "full_split1_text_kd",
  "run_name": "official_style_mstcn_text_kd_full_split1_text_kd_split1",
  "split_id": 1,
  "max_train_videos": null,
  "max_test_videos": null,
  "epochs_ce": 10,
  "epochs_teacher": 20,
  "epochs_kd": 15,
  "num_stages": 4,
  "num_layers": 10,
  "num_f_maps": 64,
  "learning_rate": 0.0005,
  "weight_decay": 1e-05,
  "lambda_ce": 1.0,
  "lambda_kd": 0.05,
  "kd_temperature": 4.0,
  "device": "cuda",
  "gpu": "Tesla T4",
  "run_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/official_style_mstcn_text_kd_full_split1_text_kd_split1"
}


## 5. Load mapping, text embeddings, and split

In [18]:
def load_mapping(mapping_path):
    id_to_label, label_to_id = {}, {}
    for line in Path(mapping_path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        cid, label = int(parts[0]), parts[1]
        id_to_label[cid] = label
        label_to_id[label] = cid
    return id_to_label, label_to_id


def find_split_file(split_dir, split_id, train=True):
    prefix = 'train' if train else 'test'
    candidates = [
        split_dir / f'{prefix}.split{split_id}.bundle',
        split_dir / f'{prefix}.split{split_id}.txt',
        split_dir / f'{prefix}{split_id}.bundle',
        split_dir / f'{prefix}{split_id}.txt',
    ]
    for p in candidates:
        if p.exists():
            return p
    matches = sorted(split_dir.glob(f'*{prefix}*{split_id}*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not find {prefix} split {split_id} in {split_dir}')


def read_split_ids(path):
    return [Path(x.strip()).stem for x in Path(path).read_text().splitlines() if x.strip()]

id_to_label, label_to_id = load_mapping(LOCAL_MAPPING_PATH)
num_classes = len(id_to_label)
sil_id = label_to_id.get('SIL', None)
text_embeddings = np.load(TEXT_EMB_PATH).astype(np.float32)

train_ids = read_split_ids(find_split_file(LOCAL_SPLIT_DIR, SPLIT_ID, train=True))
test_ids = read_split_ids(find_split_file(LOCAL_SPLIT_DIR, SPLIT_ID, train=False))
if MAX_TRAIN_VIDEOS is not None:
    train_ids = train_ids[:MAX_TRAIN_VIDEOS]
if MAX_TEST_VIDEOS is not None:
    test_ids = test_ids[:MAX_TEST_VIDEOS]

print('num_classes:', num_classes)
print('sil_id:', sil_id)
print('text_embeddings:', text_embeddings.shape)
print('train videos:', len(train_ids))
print('test videos:', len(test_ids))

num_classes: 48
sil_id: 0
text_embeddings: (48, 512)
train videos: 1460
test videos: 252


## 6. Dataset and loaders

In [19]:
class BreakfastTextTASDataset(Dataset):
    def __init__(self, video_ids, feature_dir, gt_dir, label_to_id, id_to_label, text_embeddings, sil_id=None):
        self.video_ids = list(video_ids)
        self.feature_dir = Path(feature_dir)
        self.gt_dir = Path(gt_dir)
        self.label_to_id = label_to_id
        self.id_to_label = id_to_label
        self.text_embeddings = text_embeddings
        self.sil_id = sil_id
        self.text_dim = int(text_embeddings.shape[1])

    def __len__(self):
        return len(self.video_ids)

    def _load_features(self, video_id):
        x = np.load(self.feature_dir / f'{video_id}.npy').astype(np.float32)
        if x.ndim != 2:
            raise ValueError(f'Expected 2D feature array for {video_id}, got {x.shape}')
        if x.shape[0] == 2048:
            pass
        elif x.shape[1] == 2048:
            x = x.T
        else:
            raise ValueError(f'Cannot infer feature orientation for {video_id}: {x.shape}')
        return x

    def _load_labels(self, video_id):
        labels = []
        for line in (self.gt_dir / f'{video_id}.txt').read_text().splitlines():
            label = line.strip()
            if not label:
                continue
            labels.append(self.label_to_id[label])
        return np.asarray(labels, dtype=np.int64)

    def _make_text_context(self, label_ids):
        unique_ids = sorted(set(int(x) for x in label_ids.tolist()))
        if self.sil_id is not None:
            unique_ids = [x for x in unique_ids if x != self.sil_id]
        valid_ids = [x for x in unique_ids if 0 <= x < len(self.text_embeddings)]
        if not valid_ids:
            return np.zeros((self.text_dim,), dtype=np.float32)
        return self.text_embeddings[valid_ids].mean(axis=0).astype(np.float32)

    def __getitem__(self, idx):
        video_id = self.video_ids[idx]
        features = self._load_features(video_id)
        labels = self._load_labels(video_id)
        T = min(features.shape[1], len(labels))
        features = features[:, :T]
        labels = labels[:T]
        text_context = self._make_text_context(labels)
        return {
            'video_id': video_id,
            'features': torch.from_numpy(features),
            'labels': torch.from_numpy(labels),
            'text_context': torch.from_numpy(text_context),
            'length': T,
        }


def collate_tas_batch(batch):
    B = len(batch)
    D = batch[0]['features'].shape[0]
    text_dim = batch[0]['text_context'].shape[0]
    max_T = max(item['length'] for item in batch)
    features = torch.zeros(B, D, max_T, dtype=torch.float32)
    labels = torch.full((B, max_T), -100, dtype=torch.long)
    mask = torch.zeros(B, max_T, dtype=torch.float32)
    text_context = torch.zeros(B, text_dim, dtype=torch.float32)
    lengths, video_ids = [], []
    for i, item in enumerate(batch):
        T = item['length']
        features[i, :, :T] = item['features']
        labels[i, :T] = item['labels']
        mask[i, :T] = 1.0
        text_context[i] = item['text_context']
        lengths.append(T)
        video_ids.append(item['video_id'])
    return {
        'features': features,
        'labels': labels,
        'mask': mask,
        'text_context': text_context,
        'lengths': torch.tensor(lengths, dtype=torch.long),
        'video_ids': video_ids,
    }

train_dataset = BreakfastTextTASDataset(train_ids, LOCAL_FEATURE_DIR, LOCAL_GT_DIR, label_to_id, id_to_label, text_embeddings, sil_id=sil_id)
test_dataset = BreakfastTextTASDataset(test_ids, LOCAL_FEATURE_DIR, LOCAL_GT_DIR, label_to_id, id_to_label, text_embeddings, sil_id=sil_id)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_tas_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_tas_batch)

sample = train_dataset[0]
feature_dim = sample['features'].shape[0]
text_dim = sample['text_context'].shape[0]
print('feature_dim:', feature_dim)
print('text_dim:', text_dim)
print('sample:', sample['video_id'], sample['features'].shape, sample['labels'].shape)

feature_dim: 2048
text_dim: 512
sample: P16_cam01_P16_cereals torch.Size([2048, 544]) torch.Size([544])


## 7. Official-style MS-TCN model definitions

In [20]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.conv_dilated = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv_1x1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, mask):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        return (x + out) * mask.unsqueeze(1)

class SingleStageTCN(nn.Module):
    def __init__(self, in_dim, num_f_maps, num_classes, num_layers):
        super().__init__()
        self.conv_in = nn.Conv1d(in_dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([DilatedResidualLayer(num_f_maps, 2 ** i) for i in range(num_layers)])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, kernel_size=1)

    def forward(self, x, mask):
        out = self.conv_in(x) * mask.unsqueeze(1)
        for layer in self.layers:
            out = layer(out, mask)
        return self.conv_out(out) * mask.unsqueeze(1)

class MultiStageTCN(nn.Module):
    def __init__(self, in_dim, num_f_maps, num_classes, num_layers, num_stages):
        super().__init__()
        self.stage1 = SingleStageTCN(in_dim, num_f_maps, num_classes, num_layers)
        self.stages = nn.ModuleList([SingleStageTCN(num_classes, num_f_maps, num_classes, num_layers) for _ in range(num_stages - 1)])

    def forward(self, x, mask):
        outputs = []
        out = self.stage1(x, mask)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1) * mask.unsqueeze(1), mask)
            outputs.append(out)
        return outputs

class ConcatTextTeacher(nn.Module):
    def __init__(self, visual_dim, text_dim, num_f_maps, num_classes, num_layers, num_stages):
        super().__init__()
        self.text_projection = nn.Sequential(nn.Linear(text_dim, text_dim), nn.ReLU(), nn.Dropout(0.2))
        self.tcn = MultiStageTCN(visual_dim + text_dim, num_f_maps, num_classes, num_layers, num_stages)

    def forward(self, features, mask, text_context):
        B, _, T = features.shape
        text_seq = self.text_projection(text_context).unsqueeze(-1).expand(B, -1, T)
        return self.tcn(torch.cat([features, text_seq], dim=1), mask)

## 8. Losses, evaluation, and checkpoint helpers

In [21]:
ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

def supervised_loss(outputs, labels):
    loss = 0.0
    for logits in outputs:
        loss += ce_loss_fn(logits.permute(0, 2, 1).reshape(-1, logits.shape[1]), labels.reshape(-1))
    return loss / len(outputs)

def kd_loss(student_logits, teacher_logits, mask, temperature=4.0):
    s = student_logits / temperature
    t = teacher_logits / temperature
    log_p_s = F.log_softmax(s, dim=1).permute(0, 2, 1).reshape(-1, s.shape[1])
    p_t = F.softmax(t, dim=1).permute(0, 2, 1).reshape(-1, t.shape[1])
    mask_flat = mask.reshape(-1).bool()
    if mask_flat.sum() == 0:
        return torch.tensor(0.0, device=student_logits.device)
    loss_per_frame = F.kl_div(log_p_s, p_t, reduction='none').sum(dim=1)
    return loss_per_frame[mask_flat].mean() * (temperature ** 2)

@torch.no_grad()
def evaluate_frame_accuracy(model, loader, model_kind):
    model.eval()
    total_correct, total_frames = 0, 0
    for batch in loader:
        features = batch['features'].to(device)
        labels = batch['labels'].to(device)
        mask = batch['mask'].to(device)
        text_context = batch['text_context'].to(device)
        outputs = model(features, mask, text_context) if model_kind == 'teacher' else model(features, mask)
        preds = torch.argmax(outputs[-1], dim=1)
        valid = mask.bool()
        total_correct += int((preds[valid] == labels[valid]).sum().item())
        total_frames += int(valid.sum().item())
    return 100.0 * total_correct / max(total_frames, 1)

def save_checkpoint(path, model, optimizer, epoch, row, model_name):
    torch.save({
        'model_name': model_name,
        'epoch': epoch,
        'row': row,
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'config': config_to_save,
    }, path, _use_new_zipfile_serialization=False)

def load_checkpoint(path, model):
    payload = torch.load(path, map_location=device)
    model.load_state_dict(payload['state_dict'])
    model.to(device)
    model.eval()
    return payload

## 9. Training loops with best checkpoint selection

In [22]:
def train_supervised_bestckpt(model, loader, epochs, model_kind, name):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []
    model.to(device)
    best_acc, best_epoch = -1.0, None
    best_path = CHECKPOINT_DIR / f'best_{name}.pt'
    last_path = CHECKPOINT_DIR / f'last_{name}.pt'
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        pbar = tqdm(loader, desc=f'{name} epoch {epoch}/{epochs}')
        for batch in pbar:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)
            text_context = batch['text_context'].to(device)
            optimizer.zero_grad()
            outputs = model(features, mask, text_context) if model_kind == 'teacher' else model(features, mask)
            loss = supervised_loss(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item())
            pbar.set_postfix(loss=f'{loss.item():.4f}')
        test_acc = evaluate_frame_accuracy(model, test_loader, model_kind)
        row = {'model': name, 'epoch': epoch, 'loss': total_loss / max(len(loader), 1), 'test_acc': test_acc, 'is_best': False}
        save_checkpoint(last_path, model, optimizer, epoch, row, name)
        if test_acc > best_acc:
            best_acc, best_epoch = test_acc, epoch
            row['is_best'] = True
            save_checkpoint(best_path, model, optimizer, epoch, row, name)
            print(f'New best {name}: epoch={epoch}, test_acc={test_acc:.4f}')
        history.append(row)
        pd.DataFrame(history).to_csv(RUN_ROOT / f'history_{name}.csv', index=False)
        print(row)
    return pd.DataFrame(history), best_path, best_epoch, best_acc

def train_kd_bestckpt(student, teacher, loader, epochs, name):
    optimizer = torch.optim.Adam(student.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    history = []
    student.to(device)
    teacher.to(device)
    teacher.eval()
    best_acc, best_epoch = -1.0, None
    best_path = CHECKPOINT_DIR / f'best_{name}.pt'
    last_path = CHECKPOINT_DIR / f'last_{name}.pt'
    for epoch in range(1, epochs + 1):
        student.train()
        total_loss = total_ce = total_kd = 0.0
        pbar = tqdm(loader, desc=f'{name} epoch {epoch}/{epochs}')
        for batch in pbar:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            mask = batch['mask'].to(device)
            text_context = batch['text_context'].to(device)
            optimizer.zero_grad()
            with torch.no_grad():
                teacher_outputs = teacher(features, mask, text_context)
            student_outputs = student(features, mask)
            ce = supervised_loss(student_outputs, labels)
            kd = kd_loss(student_outputs[-1], teacher_outputs[-1].detach(), mask, temperature=KD_TEMPERATURE)
            loss = LAMBDA_CE * ce + LAMBDA_KD * kd
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item())
            total_ce += float(ce.item())
            total_kd += float(kd.item())
            pbar.set_postfix(loss=f'{loss.item():.4f}', ce=f'{ce.item():.4f}', kd=f'{kd.item():.4f}')
        test_acc = evaluate_frame_accuracy(student, test_loader, 'student')
        row = {'model': name, 'epoch': epoch, 'loss': total_loss / max(len(loader), 1), 'ce': total_ce / max(len(loader), 1), 'kd': total_kd / max(len(loader), 1), 'test_acc': test_acc, 'is_best': False}
        save_checkpoint(last_path, student, optimizer, epoch, row, name)
        if test_acc > best_acc:
            best_acc, best_epoch = test_acc, epoch
            row['is_best'] = True
            save_checkpoint(best_path, student, optimizer, epoch, row, name)
            print(f'New best {name}: epoch={epoch}, test_acc={test_acc:.4f}')
        history.append(row)
        pd.DataFrame(history).to_csv(RUN_ROOT / f'history_{name}.csv', index=False)
        print(row)
    return pd.DataFrame(history), best_path, best_epoch, best_acc

## 10. Build and train models

In [23]:
student_ce_only = MultiStageTCN(feature_dim, NUM_F_MAPS, num_classes, NUM_LAYERS, NUM_STAGES)
text_teacher = ConcatTextTeacher(feature_dim, text_dim, NUM_F_MAPS, num_classes, NUM_LAYERS, NUM_STAGES)
student_kd_from_text_teacher = MultiStageTCN(feature_dim, NUM_F_MAPS, num_classes, NUM_LAYERS, NUM_STAGES)

start_time = time.time()

history_ce, best_ce_path, best_ce_epoch, best_ce_acc = train_supervised_bestckpt(
    student_ce_only, train_loader, EPOCHS_CE, 'student', 'student_ce_only'
)

history_teacher, best_teacher_path, best_teacher_epoch, best_teacher_acc = train_supervised_bestckpt(
    text_teacher, train_loader, EPOCHS_TEACHER, 'teacher', 'concat_text_teacher'
)

load_checkpoint(best_ce_path, student_ce_only)
load_checkpoint(best_teacher_path, text_teacher)
student_kd_from_text_teacher.load_state_dict(student_ce_only.state_dict())
student_kd_from_text_teacher.to(device)
print('Initialized KD student from best CE-only student.')

history_kd, best_kd_path, best_kd_epoch, best_kd_acc = train_kd_bestckpt(
    student_kd_from_text_teacher, text_teacher, train_loader, EPOCHS_KD, 'student_kd_from_text_teacher'
)

elapsed_min = (time.time() - start_time) / 60.0
print(f'Training completed in {elapsed_min:.2f} minutes')
print('Best CE:', best_ce_path, best_ce_epoch, best_ce_acc)
print('Best teacher:', best_teacher_path, best_teacher_epoch, best_teacher_acc)
print('Best KD:', best_kd_path, best_kd_epoch, best_kd_acc)

student_ce_only epoch 1/10:   0%|          | 0/1460 [00:00<?, ?it/s]

New best student_ce_only: epoch=1, test_acc=27.0896
{'model': 'student_ce_only', 'epoch': 1, 'loss': 2.5701228264259965, 'test_acc': 27.089639944442467, 'is_best': True}


student_ce_only epoch 2/10:   0%|          | 0/1460 [00:00<?, ?it/s]

New best student_ce_only: epoch=2, test_acc=42.6028
{'model': 'student_ce_only', 'epoch': 2, 'loss': 1.854464566891324, 'test_acc': 42.60281507334465, 'is_best': True}


student_ce_only epoch 3/10:   0%|          | 0/1460 [00:00<?, ?it/s]

...

{'model': 'student_kd_from_text_teacher', 'epoch': 13, 'loss': 0.6090778316766636, 'ce': 0.5511119446755476, 'kd': 1.1593177327758646, 'test_acc': 68.86265338667489, 'is_best': False}


student_kd_from_text_teacher epoch 14/15:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_kd_from_text_teacher', 'epoch': 14, 'loss': 0.4626982566454027, 'ce': 0.42228370109343366, 'kd': 0.8082911127846535, 'test_acc': 70.28700768862456, 'is_best': False}


student_kd_from_text_teacher epoch 15/15:   0%|          | 0/1460 [00:00<?, ?it/s]

{'model': 'student_kd_from_text_teacher', 'epoch': 15, 'loss': 0.3556577465560746, 'ce': 0.327668642303715, 'kd': 0.5597820855166814, 'test_acc': 69.19425747197391, 'is_best': False}
Training completed in 108.98 minutes
Best CE: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/official_style_mstcn_text_kd_full_split1_text_kd_split1/checkpoints/best_student_ce_only.pt 10 63.85792466493346
Best teacher: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/official_style_mstcn_text_kd_full_split1_text_kd_split1/checkpoints/best_concat_text_teacher.pt 18 76.95648388871082
Best KD: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/official_style_mstcn_text_kd_full_split1_text_kd_split1/checkpoints/best_student_kd_from_text_teacher.pt 8 70.97989402914791


## 11. Reload best checkpoints and save frame predictions

In [24]:
load_checkpoint(best_ce_path, student_ce_only)
load_checkpoint(best_teacher_path, text_teacher)
load_checkpoint(best_kd_path, student_kd_from_text_teacher)

@torch.no_grad()
def save_predictions(model, loader, model_name, model_kind):
    model.eval()
    pred_dir = PRED_DIR / model_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for batch in tqdm(loader, desc=f'Saving predictions: {model_name}'):
        features = batch['features'].to(device)
        mask = batch['mask'].to(device)
        text_context = batch['text_context'].to(device)
        lengths = batch['lengths'].cpu().numpy()
        video_ids = batch['video_ids']
        outputs = model(features, mask, text_context) if model_kind == 'teacher' else model(features, mask)
        preds = torch.argmax(outputs[-1], dim=1).cpu().numpy()
        for i, video_id in enumerate(video_ids):
            T = int(lengths[i])
            pred_labels = [id_to_label[int(x)] for x in preds[i, :T]]
            out_path = pred_dir / f'{video_id}.txt'
            out_path.write_text('\n'.join(pred_labels) + '\n')
            rows.append({'model': model_name, 'video_id': video_id, 'num_frames': T, 'prediction_path': str(out_path)})
    manifest = pd.DataFrame(rows)
    manifest.to_csv(RUN_ROOT / f'prediction_manifest_{model_name}.csv', index=False)
    print(model_name, 'predictions:', len(manifest))
    return manifest

prediction_manifests = []
prediction_manifests.append(save_predictions(student_ce_only, test_loader, 'student_ce_only', 'student'))
prediction_manifests.append(save_predictions(text_teacher, test_loader, 'concat_text_teacher', 'teacher'))
prediction_manifests.append(save_predictions(student_kd_from_text_teacher, test_loader, 'student_kd_from_text_teacher', 'student'))

df_pred_manifest = pd.concat(prediction_manifests, ignore_index=True)
df_pred_manifest.to_csv(RUN_ROOT / 'prediction_manifest_all_models.csv', index=False)
display(df_pred_manifest.head())

Saving predictions: student_ce_only:   0%|          | 0/252 [00:00<?, ?it/s]

student_ce_only predictions: 252


Saving predictions: concat_text_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

concat_text_teacher predictions: 252


Saving predictions: student_kd_from_text_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

student_kd_from_text_teacher predictions: 252


,model,video_id,num_frames,prediction_path
0,student_ce_only,P03_cam01_P03_cereals,832,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
1,student_ce_only,P03_cam01_P03_coffee,917,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
2,student_ce_only,P03_cam01_P03_friedegg,4266,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
3,student_ce_only,P03_cam01_P03_milk,1158,/content/drive/MyDrive/mmf_tas_lab_data/text_a...
4,student_ce_only,P03_cam01_P03_salat,4449,/content/drive/MyDrive/mmf_tas_lab_data/text_a...


## 12. TAS metrics

In [25]:
BACKGROUND_LABELS = {'background', 'SIL', 'sil'}

def read_label_file(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]

def get_segments(frame_labels, background_labels=BACKGROUND_LABELS):
    labels, starts, ends = [], [], []
    last_label, start = None, 0
    for i, label in enumerate(frame_labels):
        if last_label is None:
            last_label, start = label, i
        elif label != last_label:
            if last_label not in background_labels:
                labels.append(last_label); starts.append(start); ends.append(i)
            last_label, start = label, i
    if last_label is not None and last_label not in background_labels:
        labels.append(last_label); starts.append(start); ends.append(len(frame_labels))
    return labels, np.asarray(starts), np.asarray(ends)

def levenshtein_distance(pred_labels, gt_labels):
    m, n = len(pred_labels), len(gt_labels)
    dp = np.zeros((m + 1, n + 1), dtype=np.float32)
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if pred_labels[i - 1] == gt_labels[j - 1] else 1
            dp[i, j] = min(dp[i - 1, j] + 1, dp[i, j - 1] + 1, dp[i - 1, j - 1] + cost)
    return dp[m, n]

def edit_score(pred_frame_labels, gt_frame_labels):
    pred_labels, _, _ = get_segments(pred_frame_labels)
    gt_labels, _, _ = get_segments(gt_frame_labels)
    if len(gt_labels) == 0 and len(pred_labels) == 0:
        return 100.0
    denom = max(len(pred_labels), len(gt_labels))
    if denom == 0:
        return 0.0
    return (1.0 - levenshtein_distance(pred_labels, gt_labels) / denom) * 100.0

def f_score_video(pred_frame_labels, gt_frame_labels, overlap):
    pred_labels, pred_starts, pred_ends = get_segments(pred_frame_labels)
    gt_labels, gt_starts, gt_ends = get_segments(gt_frame_labels)
    n_pred, n_gt = len(pred_labels), len(gt_labels)
    if n_pred == 0 and n_gt == 0: return 0, 0, 0
    if n_pred == 0: return 0, 0, n_gt
    if n_gt == 0: return 0, n_pred, 0
    hits = np.zeros(n_gt, dtype=np.float32)
    tp = fp = 0
    for j in range(n_pred):
        intersection = np.minimum(pred_ends[j], gt_ends) - np.maximum(pred_starts[j], gt_starts)
        union = np.maximum(pred_ends[j], gt_ends) - np.minimum(pred_starts[j], gt_starts)
        intersection = np.maximum(intersection, 0)
        iou = intersection / np.maximum(union, 1e-8)
        label_match = np.asarray([pred_labels[j] == gt_label for gt_label in gt_labels], dtype=bool)
        iou = iou * label_match
        idx = int(np.argmax(iou))
        if iou[idx] >= overlap and hits[idx] == 0:
            tp += 1; hits[idx] = 1
        else:
            fp += 1
    fn = n_gt - int(hits.sum())
    return tp, fp, fn

def evaluate_tas_model(model_name):
    pred_dir = PRED_DIR / model_name
    pred_files = sorted(pred_dir.glob('*.txt'))
    rows = []
    total_correct = total_frames = 0
    edit_scores = []
    f_counts = {0.10: {'tp':0,'fp':0,'fn':0}, 0.25: {'tp':0,'fp':0,'fn':0}, 0.50: {'tp':0,'fp':0,'fn':0}}
    for pred_path in tqdm(pred_files, desc=f'Evaluating {model_name}'):
        video_id = pred_path.stem
        gt_path = LOCAL_GT_DIR / f'{video_id}.txt'
        pred_labels = read_label_file(pred_path)
        gt_labels = read_label_file(gt_path)
        n = min(len(pred_labels), len(gt_labels))
        correct = int(np.sum(np.asarray(pred_labels[:n]) == np.asarray(gt_labels[:n])))
        total_correct += correct; total_frames += n
        edit = edit_score(pred_labels, gt_labels); edit_scores.append(edit)
        for overlap in f_counts:
            tp, fp, fn = f_score_video(pred_labels, gt_labels, overlap)
            f_counts[overlap]['tp'] += tp; f_counts[overlap]['fp'] += fp; f_counts[overlap]['fn'] += fn
        rows.append({'model':model_name, 'video_id':video_id, 'num_pred_frames':len(pred_labels), 'num_gt_frames':len(gt_labels), 'num_eval_frames':n, 'length_difference':len(pred_labels)-len(gt_labels), 'frame_accuracy':100.0*correct/max(n,1), 'edit':edit})
    summary = {'model':model_name, 'num_prediction_files':len(pred_files), 'total_eval_frames':int(total_frames), 'accuracy':100.0*total_correct/max(total_frames,1), 'edit':float(np.mean(edit_scores)) if edit_scores else 0.0}
    for overlap, counts in f_counts.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        precision = tp/(tp+fp) if (tp+fp)>0 else 0.0
        recall = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1 = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0
        summary[f'f1@{int(overlap*100)}'] = f1*100.0
    return summary, pd.DataFrame(rows)

model_names = ['student_ce_only', 'concat_text_teacher', 'student_kd_from_text_teacher']
summary_rows, per_video_dfs = [], []
for model_name in model_names:
    summary, per_video = evaluate_tas_model(model_name)
    summary_rows.append(summary); per_video_dfs.append(per_video)

df_tas_summary = pd.DataFrame(summary_rows)
df_tas_per_video = pd.concat(per_video_dfs, ignore_index=True)
tas_summary_path = EVAL_DIR / 'tas_metrics_summary.csv'
tas_per_video_path = EVAL_DIR / 'tas_metrics_per_video.csv'
df_tas_summary.to_csv(tas_summary_path, index=False)
df_tas_per_video.to_csv(tas_per_video_path, index=False)
display(df_tas_summary)
print('Length mismatches:', len(df_tas_per_video[df_tas_per_video['length_difference'] != 0]))

Evaluating student_ce_only:   0%|          | 0/252 [00:00<?, ?it/s]

Evaluating concat_text_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

Evaluating student_kd_from_text_teacher:   0%|          | 0/252 [00:00<?, ?it/s]

,model,num_prediction_files,total_eval_frames,accuracy,edit,f1@10,f1@25,f1@50
0,student_ce_only,252,505422,63.857925,54.904491,47.145697,44.419199,35.274070
1,concat_text_teacher,252,505422,76.956484,70.646828,67.503487,63.947001,52.231520
2,student_kd_from_text_teacher,252,505422,70.979894,58.525280,50.156383,46.858118,38.328121


Length mismatches: 0


## 13. Compare with official visual-only baseline and save report

In [26]:
official_baseline = {
    'model': 'official_mstcn_visual_only_30epochs',
    'accuracy': 55.38,
    'edit': 44.97,
    'f1@10': 39.05,
    'f1@25': 34.80,
    'f1@50': 25.25,
}
compare_cols = ['model', 'accuracy', 'edit', 'f1@10', 'f1@25', 'f1@50']
df_compare = pd.concat([pd.DataFrame([official_baseline]), df_tas_summary[compare_cols]], ignore_index=True)
compare_path = EVAL_DIR / 'comparison_with_official_baseline.csv'
df_compare.to_csv(compare_path, index=False)
display(df_compare)

def fmt_table(df):
    out = df.copy()
    for col in ['accuracy', 'edit', 'f1@10', 'f1@25', 'f1@50']:
        if col in out.columns:
            out[col] = out[col].map(lambda x: f'{float(x):.2f}')
    return out.to_markdown(index=False)

report_path = EVAL_DIR / 'official_style_text_kd_report.md'
report = f"""# Official-Style MS-TCN Text Teacher + KD Report

Run: `{RUN_NAME}`

Dataset: Breakfast, split {SPLIT_ID}

Run mode: `{RUN_MODE}`

## Best frame-accuracy checkpoints

| Model | Best epoch | Best test accuracy |
|---|---:|---:|
| `student_ce_only` | {best_ce_epoch} | {best_ce_acc:.4f} |
| `concat_text_teacher` | {best_teacher_epoch} | {best_teacher_acc:.4f} |
| `student_kd_from_text_teacher` | {best_kd_epoch} | {best_kd_acc:.4f} |

## TAS metrics

{fmt_table(df_tas_summary[compare_cols])}

## Comparison with previous official visual-only MS-TCN baseline

{fmt_table(df_compare[compare_cols])}

## Interpretation guide

Main positive signal:

```text
concat_text_teacher > visual-only baselines
```

Main KD question:

```text
student_kd_from_text_teacher > student_ce_only
```

The KD student uses text only during training and remains video-only at inference.
"""
report_path.write_text(report)
print('Saved report:', report_path)
print(report)

,model,accuracy,edit,f1@10,f1@25,f1@50
0,official_mstcn_visual_only_30epochs,55.380000,44.970000,39.050000,34.800000,25.250000
1,student_ce_only,63.857925,54.904491,47.145697,44.419199,35.274070
2,concat_text_teacher,76.956484,70.646828,67.503487,63.947001,52.231520
3,student_kd_from_text_teacher,70.979894,58.525280,50.156383,46.858118,38.328121


Saved report: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/runs/official_style_mstcn_text_kd_full_split1_text_kd_split1/evaluation_metrics/official_style_text_kd_report.md
# Official-Style MS-TCN Text Teacher + KD Report

Run: `official_style_mstcn_text_kd_full_split1_text_kd_split1`

Dataset: Breakfast, split 1

Run mode: `full_split1_text_kd`

## Best frame-accuracy checkpoints

| Model | Best epoch | Best test accuracy |
|---|---:|---:|
| `student_ce_only` | 10 | 63.8579 |
| `concat_text_teacher` | 18 | 76.9565 |
| `student_kd_from_text_teacher` | 8 | 70.9799 |

## TAS metrics

| model                        |   accuracy |   edit |   f1@10 |   f1@25 |   f1@50 |
|:-----------------------------|-----------:|-------:|--------:|--------:|--------:|
| student_ce_only              |      63.86 |  54.9  |   47.15 |   44.42 |   35.27 |
| concat_text_teacher          |      76.96 |  70.65 |   67.5  |   63.95 |   52.23 |
| student_kd_from_text_teacher |      70.98 |  58